# Autorzy: Konrad Małek, Jan Jędra, Mikołaj Kołek

# Zadanie 3: optymalizacja dyskretna

Termin realizacji: 20 kwietnia 2026

Zadanie do oddania przez MS Teams. Do oddania: kod oraz krótkie sprawozdanie w PDF (można na przykład przy użyciu `quarto render notebook.ipynb --to pdf`).

## Na 3.0

Do realizacji:

1. Zaimplementuj dyskretny problem plecakowy z trzema plecakami w MiniZinc na podstawie przykładu (plik `minizinc.ipynb`). Spróbuj rozwiązać problem dla 10 zestawów parametrów o różnych wielkościach tak, aby rozwiązanie największego problemu trwało powyżej 5 sekund. Zanotuj w każdym przypadku liczbę wszystkich przedmiotów, pojemności plecaków, liczbę wybranych przedmiotów i sumaryczną wartość przedmiotów w każdym plecaku osobno.
2. Losowanie przedmiotów w każdym przypadku powinno być tak ustawione, aby optymalne rozwiązanie wymagało wzięcia przynajmniej dwóch przedmiotów do każdego plecaka oraz zostawienia przynajmniej dwóch przedmiotów poza plecakami. W raporcie należy umieścić informację w jaki sposób zostało to zweryfikowane.
3. Zmodyfikuj metodę z notatnika `tabu_search.ipynb` tak aby rozwiązywała opisywany problem plecakowy. Porównaj na tych samych problemach czy Minizinc i Tabu search zwracają równie dobre rozwiazania, oraz wypisz jakie to są rozwiązania. Wykonaj eksperymenty z trzema różnymi długościami listy zakazów (1, 2, 5).

## Na 4.0

Do realizacji:

1. Punkty z zadania na 3.0.
2. Rozszerz możliwe ruchy w tabu search o przeniesienie przedmiotu z jednego plecaka do drugiego. Zapisz rozważ czy to poprawia działanie metody (czy znalezione jest lepsze, takie samo czy gorsze rozwiązanie? czy rozwiązanie jest znajdowane szybciej czy wolniej?). Dla każdego z 10 zestawów parametrów problemu plecakowego wykonaj ocenę przez uśrednienie dla 10 różnych losowych przypadków.
3. Podsumuj dane w formie tabelki z czterema kolumnami (Minizinc, tabu search z listą o długości 1, 2, i 5) oraz 10 wierszami (po jednym dla zestawu parametrów problemu), a w komórkach umieść średnią wartość wartości przedmiotów oraz średni czas potrzebny do uzyskania rozwiązania.

## Na 5.0

Do realizacji:

1. Punkty z zadania na 4.0.
2. Zaimplementuj samodzielnie algorytm symulowanego wyżarzania z podobnym interfejsem co tabu search. Porównaj jego działanie do rozważanych wcześniej rozwiązań dla trzech różnych schematów chłodzenia. Dobierz liczbę iteracji tak, aby czas działania był porównywalny do tabu search.


## Realizacja na 3.0

In [ ]:
using Random
using JSON
using Printf
using Statistics
using DataStructures

const LAB_DIR = @__DIR__
const MODEL_PATH = joinpath(LAB_DIR, "knapsack_3.mzn")

# 10 rozmiarów instancji, czyli od 10 do 55 przedmiotow
const SIZES = [10, 15, 20, 25, 30, 35, 40, 45, 50, 55]

# Limity czasu i liczby iteracji
const MZ_LIMIT_SECTION30_MS = 6000
const MZ_LIMIT_SECTION40_MS = 2000
const TS_ITER_SECTION30 = 400
const TS_ITER_SECTION40 = 500
const AVG_RUNS = 10

# template modelu MiniZinc
const MZN_MODEL = """
int: N;
set of int: ITEM = 1..N;

int: C1;
int: C2;
int: C3;

array[ITEM] of int: profits;
array[ITEM] of int: weights;

array[ITEM] of var 0..1: in_k1;
array[ITEM] of var 0..1: in_k2;
array[ITEM] of var 0..1: in_k3;

constraint forall(i in ITEM)(in_k1[i] + in_k2[i] + in_k3[i] <= 1);
constraint sum(i in ITEM)(weights[i] * in_k1[i]) <= C1;
constraint sum(i in ITEM)(weights[i] * in_k2[i]) <= C2;
constraint sum(i in ITEM)(weights[i] * in_k3[i]) <= C3;

solve maximize sum(i in ITEM)(profits[i] * (in_k1[i] + in_k2[i] + in_k3[i]));

output [
  "{\\"in_k1\\": ", show(in_k1), ", \\"in_k2\\": ", show(in_k2), ", \\"in_k3\\": ", show(in_k3), "}\\n"
];
"""

struct MultiKnapsackProblem
    # capacities[k] - pojemność k-tego plecaka
    # weights[i], profits[i] - waga i wartość i-tego przedmiotu
    capacities::Vector{Int}
    weights::Vector{Int}
    profits::Vector{Int}
end

struct SolverResult
    # x[i] = 0 oznacza ze przedmiot nie zostal wybrany
    x::Vector{Int}
    value::Int
    time::Float64
    status::String
end

function ensure_model_file()
    open(MODEL_PATH, "w") do io
        write(io, MZN_MODEL)
    end
end

function build_instance(n::Int; seed::Int=n)
    n < 8 && error("Instancja musi mieć co najmniej 8 przedmiotów.")
    rng = MersenneTwister(42 + seed)

    # tworzenie pozostalych przedmiotow ktore sa opcjonalne i suma 
    # ich wartosci jest mniejsza niz wartosc kazdego z 6 rdzeniowych przedmiotow
    optional_count = n - 8
    optional_weights = rand(rng, 4:35, optional_count)
    optional_profits = rand(rng, 5:110, optional_count)

    total_optional_profit = sum(optional_profits)
    core_profit = total_optional_profit + 500

    # 6 przedmiotów rdzeniowych musi wejść do plecakow
    # każdy jest cenniejszy niż wszystkie opcjonalne razem
    # 2 ciężkie przedmioty mają zostać zawsze poza plecakami
    weights = vcat(fill(60, 6), [220, 225], optional_weights)
    profits = vcat(fill(core_profit, 6), [1, 1], optional_profits)
    capacities = [170, 170, 170]

    return MultiKnapsackProblem(capacities, collect(weights), collect(profits))
end

function certificate_holds(kp::MultiKnapsackProblem)
    #  ten kawałek poprostu sprawdza czy nasz dowod jest spelniony z zadania 2
    optional_profit_sum = sum(kp.profits[9:end])
    core_profit = kp.profits[1]
    core_weight = kp.weights[1]
    max_capacity = maximum(kp.capacities)

    cond_profit = core_profit > optional_profit_sum
    cond_fit_two = all(2 * core_weight <= c for c in kp.capacities)
    cond_block_three = all(3 * core_weight > c for c in kp.capacities)
    cond_forbidden = all(kp.weights[i] > max_capacity for i in 7:8)

    ok = cond_profit && cond_fit_two && cond_block_three && cond_forbidden
    explanation = @sprintf(
        "core_profit=%d > suma_opcjonalnych=%d, 2*core=%d <= C=%d, 3*core=%d > C=%d, ciężkie=[%d,%d]",
        core_profit, optional_profit_sum, 2 * core_weight, kp.capacities[1],
        3 * core_weight, kp.capacities[1], kp.weights[7], kp.weights[8]
    )
    return ok, explanation
end

function bag_weights(kp::MultiKnapsackProblem, x::Vector{Int})
    # obliczenie wag przedmiotow w plecaku
    loads = zeros(Int, length(kp.capacities))
    for i in eachindex(x)
        if x[i] > 0
            loads[x[i]] += kp.weights[i]
        end
    end
    return loads
end

function summarize_solution(kp::MultiKnapsackProblem, x::Vector{Int})
    # zebranie statystyk do raportu
    count_k = zeros(Int, 3)
    value_k = zeros(Int, 3)
    chosen = Int[]

    for i in eachindex(x)
        k = x[i]
        if 1 <= k <= 3
            count_k[k] += 1
            value_k[k] += kp.profits[i]
            push!(chosen, i)
        end
    end

    outside = count(==(0), x)
    total_value = sum(value_k)
    loads = bag_weights(kp, x)
    return count_k, value_k, outside, total_value, loads, chosen
end

function property_holds_on_solution(x::Vector{Int})
    # funkcja sprawdzajaca warunek z punktu 2
    counts = [count(==(k), x) for k in 1:3]
    outside = count(==(0), x)
    return all(c >= 2 for c in counts) && outside >= 2
end

function objective(kp::MultiKnapsackProblem, x::Vector{Int})
    # funkcja celu
    val = 0
    for i in eachindex(x)
        if x[i] > 0
            val += kp.profits[i]
        end
    end
    return -val
end

#funkcje operacji na plecakach
function apply_move!(x::Vector{Int}, move::Tuple{Symbol, Int, Int, Int})
    kind, item, _, dst = move
    if kind === :remove
        x[item] = 0
    else
        x[item] = dst
    end
    return x
end

function invert_move(move::Tuple{Symbol, Int, Int, Int})
    kind, item, src, dst = move
    if kind === :add
        return (:remove, item, dst, 0)
    elseif kind === :remove
        return (:add, item, 0, src)
    else
        return (:transfer, item, dst, src)
    end
end

function possible_moves(kp::MultiKnapsackProblem, x::Vector{Int}; allow_transfer::Bool=false)
    moves = Tuple{Symbol, Int, Int, Int}[]
    loads = bag_weights(kp, x)

    for i in eachindex(x)
        if x[i] == 0
            for k in 1:3
                if loads[k] + kp.weights[i] <= kp.capacities[k]
                    push!(moves, (:add, i, 0, k))
                end
            end
        else
            push!(moves, (:remove, i, x[i], 0))
            if allow_transfer
                for k in 1:3
                    if k != x[i] && loads[k] + kp.weights[i] <= kp.capacities[k]
                        push!(moves, (:transfer, i, x[i], k))
                    end
                end
            end
        end
    end

    return moves
end

function randomized_initial_solution(kp::MultiKnapsackProblem; seed::Int=0)
    # początkowe heurystyczne rozwiazaine, czyli punkt startowy
    # działa tak, że sortuje po wartosci i wklada przedmioty do plecakow
    # tak dlugo az jest miejsce
    rng = MersenneTwister(10_000 + seed)
    x = zeros(Int, length(kp.weights))
    loads = zeros(Int, 3)

    order = collect(eachindex(kp.weights))
    shuffle!(rng, order)
    sort!(order; by=i -> (-kp.profits[i] / kp.weights[i], -kp.profits[i]))

    for i in order
        feasible = Int[]
        for k in 1:3
            if loads[k] + kp.weights[i] <= kp.capacities[k]
                push!(feasible, k)
            end
        end
        if !isempty(feasible)
            best_remaining = minimum(kp.capacities[k] - (loads[k] + kp.weights[i]) for k in feasible)
            tied = [k for k in feasible if kp.capacities[k] - (loads[k] + kp.weights[i]) == best_remaining]
            chosen_bag = rand(rng, tied)
            x[i] = chosen_bag
            loads[chosen_bag] += kp.weights[i]
        end
    end

    return x
end

function solve_tabu(kp::MultiKnapsackProblem;
        tabu_len::Int,
        iteration_limit::Int,
        allow_transfer::Bool,
        seed::Int)

    # Tabu Search startuje od rozwiązania początkowego i poprawia je ruchem lokalnym
    x0 = randomized_initial_solution(kp; seed=seed)
    current = copy(x0)
    current_obj = objective(kp, current)
    best = copy(current)
    best_obj = current_obj
    #lista tabu czyli zakazanych ruchow, przykladowo jesli cos dodalismy do plecaka
    # to nie mozemy w nastepnej iteracji tego usunac
    tabu = CircularBuffer{Tuple{Symbol, Int, Int, Int}}(tabu_len)

    t1 = time_ns()
    for _ in 1:iteration_limit
        moves = possible_moves(kp, current; allow_transfer=allow_transfer)
        best_move = nothing
        best_move_obj = typemax(Int)

        for move in moves
            candidate = copy(current)
            apply_move!(candidate, move)
            cand_obj = objective(kp, candidate)
            move_is_tabu = move in tabu

            # dopuszcamy ruch jesli poprawia wynik
            if move_is_tabu && cand_obj >= best_obj
                continue
            end

            if cand_obj < best_move_obj
                best_move = move
                best_move_obj = cand_obj
            end
        end

        isnothing(best_move) && break

        apply_move!(current, best_move)
        current_obj = best_move_obj
        push!(tabu, invert_move(best_move))

        if current_obj < best_obj
            best = copy(current)
            best_obj = current_obj
        end
    end
    t2 = time_ns()

    return SolverResult(best, -best_obj, (t2 - t1) / 1e9, "TABU")
end

#definicja algorytmu wyżarzania z zadania na 5.0
function solve_sa(kp::MultiKnapsackProblem;
        cooling_scheme::Symbol,
        max_iter::Int,
        allow_transfer::Bool,
        seed::Int,
        initial_temp::Float64=150.0,
        alpha::Float64=0.992)

    # Symulowane wyżarzanie: losujemy ruch, następnie zawsze akceptujemy lepsze
    # rozwiązanie i czasem akceptujemy gorsze rozwiązanie, żeby nie utknąć
    # w lokalnym optimum
    rng = MersenneTwister(20_000 + seed)
    current = randomized_initial_solution(kp; seed=seed)
    current_obj = objective(kp, current)
    best = copy(current)
    best_obj = current_obj

    temp = initial_temp
    t1 = time_ns()
    for iter in 1:max_iter
        moves = possible_moves(kp, current; allow_transfer=allow_transfer)
        isempty(moves) && break

        move = rand(rng, moves)
        candidate = copy(current)
        apply_move!(candidate, move)
        cand_obj = objective(kp, candidate)
        delta = cand_obj - current_obj

        if delta < 0 || rand(rng) < exp(-delta / max(temp, 1e-9))
            current = candidate
            current_obj = cand_obj
            if current_obj < best_obj
                best = copy(current)
                best_obj = current_obj
            end
        end

        # Trzy schematy chłodzenia wymagane w części 5.0.
        # liniowy czyli szansa na akceptacje złego rozwiązania maleje liniowo 
        # geometryczny czyli szansa na akceptacje złego rozwiązania maleje wykładniczo 
        # logarytmiczny czyli szansa na akceptacje złego rozwiązania maleje wolniej 
        # niż liniowo, ale szybciej niż geometrycznie
        if cooling_scheme === :linear
            temp = max(initial_temp * (1 - iter / max_iter), 1e-6)
        elseif cooling_scheme === :geometric
            temp = max(temp * alpha, 1e-6)
        elseif cooling_scheme === :logarithmic
            temp = max(initial_temp / log(iter + 2), 1e-6)
        else
            error("Nieznany schemat chłodzenia: $cooling_scheme")
        end
    end
    t2 = time_ns()

    return SolverResult(best, -best_obj, (t2 - t1) / 1e9, string(cooling_scheme))
end

function write_dzn(io, kp::MultiKnapsackProblem)
    # zapisywanie do dzn
    n = length(kp.weights)
    write(io, "N = $n;\n")
    write(io, "C1 = $(kp.capacities[1]);\n")
    write(io, "C2 = $(kp.capacities[2]);\n")
    write(io, "C3 = $(kp.capacities[3]);\n")
    write(io, "profits = $(kp.profits);\n")
    write(io, "weights = $(kp.weights);\n")
end

function parse_minizinc_assignment(output_str::String, n::Int)
    # parsujemy output z minizynca
    x = zeros(Int, n)
    matches = collect(eachmatch(r"\{\"in_k1\":\s*\[.*?\],\s*\"in_k2\":\s*\[.*?\],\s*\"in_k3\":\s*\[.*?\]\}"s, output_str))
    isempty(matches) && return x

    sol = JSON.parse(matches[end].match)
    for i in 1:n
        if sol["in_k1"][i] != 0
            x[i] = 1
        elseif sol["in_k2"][i] != 0
            x[i] = 2
        elseif sol["in_k3"][i] != 0
            x[i] = 3
        end
    end
    return x
end

function solve_minizinc(kp::MultiKnapsackProblem; time_limit_ms::Int)
    # uruchomienie solvera minizync
    n = length(kp.weights)
    dzn_path = tempname(LAB_DIR) * ".dzn"
    open(dzn_path, "w") do io
        write_dzn(io, kp)
    end

    cmd = `minizinc --solver gecode $MODEL_PATH $dzn_path --output-time --output-objective --time-limit $time_limit_ms -O3`
    t1 = time_ns()
    stdout_buff = Pipe()
    stderr_buff = Pipe()
    proc = run(pipeline(ignorestatus(cmd), stdout=stdout_buff, stderr=stderr_buff))
    close(stdout_buff.in)
    close(stderr_buff.in)
    output_str = String(read(stdout_buff))
    error_str = String(read(stderr_buff))
    t2 = time_ns()

    rm(dzn_path; force=true)

    if proc.exitcode != 0
        println("MiniZinc error: ", strip(error_str))
        return SolverResult(zeros(Int, n), 0, (t2 - t1) / 1e9, "ERROR")
    end

    status = if occursin("==========", output_str)
        "OPTIMAL"
    elseif occursin("=====UNKNOWN=====", output_str)
        "UNKNOWN"
    else
        "FEASIBLE"
    end

    x = parse_minizinc_assignment(output_str, n)
    _, _, _, total_value, _, _ = summarize_solution(kp, x)
    return SolverResult(x, total_value, (t2 - t1) / 1e9, status)
end

function fmt_time(x::Float64)
    return @sprintf("%.4f", x)
end

function print_solution(label::String, kp::MultiKnapsackProblem, result::SolverResult)
    # printowanie porownania wynikow
    counts, values, outside, total_value, loads, _ = summarize_solution(kp, result.x)
    println(@sprintf("  %s | V=%d | T=%ss | status=%s", label, result.value, fmt_time(result.time), result.status))
    println("    Wybrane=$(counts) | Poza=$outside | Wartości=$(values) | Obciążenia=$(loads)")
    println("    x = $(result.x)")
    println("    warunek_2+2+2+poza2 = $(property_holds_on_solution(result.x))")
end

function classify_delta(delta_v::Float64, delta_t::Float64)
    quality = if delta_v > 1e-6
        "lepsze"
    elseif delta_v < -1e-6
        "gorsze"
    else
        "takie samo"
    end

    speed = if delta_t < -1e-6
        "szybciej"
    elseif delta_t > 1e-6
        "wolniej"
    else
        "porównywalnie"
    end

    return quality, speed
end

function run_section_30()
    # Część 3.0: rozwiązania MiniZinc i weryfikacja punktu 2
    ensure_model_file()
    println("=== Rozwiązywanie 10 zestawów parametrów dla 3 plecaków (MiniZinc) ===")

    instances = Dict{Int, MultiKnapsackProblem}()
    mz_results = Dict{Int, SolverResult}()
    for s in SIZES
        kp = build_instance(s; seed=s)
        instances[s] = kp
        mz_results[s] = solve_minizinc(kp; time_limit_ms=MZ_LIMIT_SECTION30_MS)
        counts, values, outside, total_value, loads, _ = summarize_solution(kp, mz_results[s].x)
        println(@sprintf(
            "N=%3d | C=%s | Wybrane=%s | poza=%d | V=%s | suma=%d | obciążenia=%s | status=%s | T=%ss",
            s, string(kp.capacities), string(counts), outside, string(values), total_value,
            string(loads), mz_results[s].status, fmt_time(mz_results[s].time)
        ))
    end

    # uzasadnienie
    println("\n=== Weryfikacja warunku z pkt 2 ===")
    println("Warunek z punktu 2 weryfikujemy z konstrukcji instancji.")
    println("W każdej instancji tworzymy 6 specjalnych przedmiotów.")
    println("Każdy z nich ma wartość większą niż suma wartości wszystkich zwykłych przedmiotów razem.")
    println("Dlatego w rozwiązaniu optymalnym trzeba wziąć wszystkie 6 specjalnych przedmiotów.")
    println("Każdy taki przedmiot waży 60, a pojemność plecaka wynosi 170, więc do jednego plecaka mieszczą się najwyżej 2 takie przedmioty.")
    println("Skoro trzeba wziąć 6 takich przedmiotów i są 3 plecaki, to każdy plecak musi dostać co najmniej 2 przedmioty.")
    println("Dodatkowo 2 ciężkie przedmioty mają wagi 220 i 225, więc nie mieszczą się do żadnego plecaka i zawsze zostają poza plecakami.")
    for s in SIZES
        ok, explanation = certificate_holds(instances[s])
        println("N=$(s) | certyfikat=$(ok) | $explanation")
    end

    return instances, mz_results
end

function run_section_30_compare(instances::Dict{Int, MultiKnapsackProblem}, mz_results::Dict{Int, SolverResult})
    # porównanie MiniZinc z Tabu Search na tych samych danych z trzema roznymi listami zakazow
    println("=== Porównanie MiniZinc i Tabu Search (ruchy add/remove, te same instancje) ===")
    basic_tabu_results = Dict{Int, Dict{Int, SolverResult}}()

    for s in SIZES
        kp = instances[s]
        println("\nN = $s")
        print_solution("MiniZinc", kp, mz_results[s])

        basic_tabu_results[s] = Dict{Int, SolverResult}()
        for tabu_len in (1, 2, 5)
            ts = solve_tabu(kp; tabu_len=tabu_len, iteration_limit=TS_ITER_SECTION30, allow_transfer=false, seed=s)
            basic_tabu_results[s][tabu_len] = ts
            relation = ts.value == mz_results[s].value ? "==" : (ts.value > mz_results[s].value ? ">" : "<")
            print_solution("TS tabu=$(tabu_len)", kp, ts)
            println("    Porównanie jakości: TS $relation MiniZinc")
        end
    end

    return basic_tabu_results
end

function run_section_40()
    # Część 4.0: sprawdzamy, czy ruch transfer między plecakami poprawia heurystykę.
    println("=== Wpływ ruchu transfer na Tabu Search (uśrednienie po 10 instancjach) ===")
    effect_rows = String[]
    table_lines = [
        "Legenda: w komórkach podano średnią wartość V i średni czas T [s]",
        "",
        "| N | MiniZinc | TS (tabu=1) | TS (tabu=2) | TS (tabu=5) |",
        "|---|---|---|---|---|"
    ]

    results = Dict{Int, Dict{Int, NamedTuple{(:value, :time), Tuple{Float64, Float64}}}}()

    for s in SIZES
        mz_values = Float64[]
        mz_times = Float64[]
        per_tabu_basic_v = Dict(t => Float64[] for t in (1, 2, 5))
        per_tabu_basic_t = Dict(t => Float64[] for t in (1, 2, 5))
        per_tabu_transfer_v = Dict(t => Float64[] for t in (1, 2, 5))
        per_tabu_transfer_t = Dict(t => Float64[] for t in (1, 2, 5))

        for run_id in 1:AVG_RUNS
            seed = s * 100 + run_id
            kp = build_instance(s; seed=seed)

            # wyniki z transferem
            mz = solve_minizinc(kp; time_limit_ms=MZ_LIMIT_SECTION40_MS)
            push!(mz_values, mz.value)
            push!(mz_times, mz.time)

            for tabu_len in (1, 2, 5)
                basic = solve_tabu(kp; tabu_len=tabu_len, iteration_limit=TS_ITER_SECTION40, allow_transfer=false, seed=seed)
                transfer = solve_tabu(kp; tabu_len=tabu_len, iteration_limit=TS_ITER_SECTION40, allow_transfer=true, seed=seed)

                push!(per_tabu_basic_v[tabu_len], basic.value)
                push!(per_tabu_basic_t[tabu_len], basic.time)
                push!(per_tabu_transfer_v[tabu_len], transfer.value)
                push!(per_tabu_transfer_t[tabu_len], transfer.time)
            end
        end

        comparisons = String[]
        for tabu_len in (1, 2, 5)
            # porównujemy średni wynik i średni czas wersji bez transferu vs z transferem.
            avg_basic_v = mean(per_tabu_basic_v[tabu_len])
            avg_basic_t = mean(per_tabu_basic_t[tabu_len])
            avg_transfer_v = mean(per_tabu_transfer_v[tabu_len])
            avg_transfer_t = mean(per_tabu_transfer_t[tabu_len])
            delta_v = avg_transfer_v - avg_basic_v
            delta_t = avg_transfer_t - avg_basic_t
            quality, speed = classify_delta(delta_v, delta_t)
            push!(comparisons, @sprintf("tabu=%d: %s (ΔV=%.1f), %s (ΔT=%+.4fs)", tabu_len, quality, delta_v, speed, delta_t))
        end
        push!(effect_rows, "N=$(s) | " * join(comparisons, " | "))

        results[s] = Dict(
            0 => (value=mean(mz_values), time=mean(mz_times)),
            1 => (value=mean(per_tabu_transfer_v[1]), time=mean(per_tabu_transfer_t[1])),
            2 => (value=mean(per_tabu_transfer_v[2]), time=mean(per_tabu_transfer_t[2])),
            5 => (value=mean(per_tabu_transfer_v[5]), time=mean(per_tabu_transfer_t[5]))
        )

        push!(table_lines, @sprintf(
            "| %d | V=%.1f, T=%.4f | V=%.1f, T=%.4f | V=%.1f, T=%.4f | V=%.1f, T=%.4f |",
            s,
            results[s][0].value, results[s][0].time,
            results[s][1].value, results[s][1].time,
            results[s][2].value, results[s][2].time,
            results[s][5].value, results[s][5].time
        ))
    end

    foreach(println, effect_rows)
    println()
    foreach(println, table_lines)

    return results
end

function calibrate_sa_iterations(kp::MultiKnapsackProblem, target_time::Float64)
    # dobieramy liczbe iteracji algorytmu wyżarzania tak żeby jego czas był
    # zbliżony do czasu TabuSeatch
    candidates = [100, 250, 500, 1000, 2000, 4000]
    best_candidate = first(candidates)
    best_gap = Inf

    for cand in candidates
        warmup = solve_sa(kp; cooling_scheme=:geometric, max_iter=cand, allow_transfer=true, seed=999)
        gap = abs(warmup.time - target_time)
        if gap < best_gap
            best_gap = gap
            best_candidate = cand
        end
    end

    return best_candidate
end

function run_section_50(exp40_results::Dict{Int, Dict{Int, NamedTuple{(:value, :time), Tuple{Float64, Float64}}}})
    # Część 5.0: porównanie algorytmu wyżarzania z wcześniejszymi metodami.
    println("=== Symulowane wyżarzanie: porównanie z wcześniejszymi metodami ===")
    println("Dla każdego rozmiaru dobieram liczbę iteracji tak, aby czas SA był zbliżony do czasu TS z listą tabu długości 5.")

    schemes = [:linear, :geometric, :logarithmic]
    table_lines = [
        "| N | iteracje SA | MiniZinc | TS tabu=5 | SA linear | SA geometric | SA logarithmic |",
        "|---|---|---|---|---|---|---|"
    ]

    for s in SIZES
        sample_kp = build_instance(s; seed=s * 100 + 1)
        target_time = exp40_results[s][5].time
        sa_iters = calibrate_sa_iterations(sample_kp, target_time)
        println(@sprintf("N=%d | target TS(5)=%.4fs | dobrane iteracje SA=%d", s, target_time, sa_iters))

        scheme_values = Dict(sym => Float64[] for sym in schemes)
        scheme_times = Dict(sym => Float64[] for sym in schemes)

        for run_id in 1:AVG_RUNS
            seed = s * 100 + run_id
            kp = build_instance(s; seed=seed)
            for scheme in schemes
                # testujemy trzy wymagane schematy chłodzenia.
                sa = solve_sa(kp; cooling_scheme=scheme, max_iter=sa_iters, allow_transfer=true, seed=seed)
                push!(scheme_values[scheme], sa.value)
                push!(scheme_times[scheme], sa.time)
            end
        end

        push!(table_lines, @sprintf(
            "| %d | %d | V=%.1f, T=%.4f | V=%.1f, T=%.4f | V=%.1f, T=%.4f | V=%.1f, T=%.4f | V=%.1f, T=%.4f |",
            s, sa_iters,
            exp40_results[s][0].value, exp40_results[s][0].time,
            exp40_results[s][5].value, exp40_results[s][5].time,
            mean(scheme_values[:linear]), mean(scheme_times[:linear]),
            mean(scheme_values[:geometric]), mean(scheme_times[:geometric]),
            mean(scheme_values[:logarithmic]), mean(scheme_times[:logarithmic])
        ))
    end

    println()
    foreach(println, table_lines)
end

section30_instances, section30_mz_results = run_section_30()


=== Rozwiązywanie 10 zestawów parametrów dla 3 plecaków (MiniZinc) ===
N= 10 | C=[170, 170, 170] | Wybrane=[2, 4, 2] | poza=2 | V=[1230, 1345, 1230] | suma=3805 | obciążenia=[120, 167, 120] | status=OPTIMAL | T=0.2010s
N= 15 | C=[170, 170, 170] | Wybrane=[4, 5, 4] | poza=2 | V=[1852, 1838, 1781] | suma=5471 | obciążenia=[161, 167, 166] | status=OPTIMAL | T=1.3053s
N= 20 | C=[170, 170, 170] | Wybrane=[5, 5, 1] | poza=9 | V=[2197, 2184, 1000] | suma=5381 | obciążenia=[170, 170, 60] | status=FEASIBLE | T=6.0408s
N= 25 | C=[170, 170, 170] | Wybrane=[9, 2, 1] | poza=13 | V=[4024, 3392, 1696] | suma=9112 | obciążenia=[170, 120, 60] | status=FEASIBLE | T=6.0393s
N= 30 | C=[170, 170, 170] | Wybrane=[7, 2, 0] | poza=21 | V=[4097, 3674, 0] | suma=7771 | obciążenia=[170, 120, 0] | status=FEASIBLE | T=6.0364s
N= 35 | C=[170, 170, 170] | Wybrane=[8, 1, 0] | poza=26 | V=[4376, 1950, 0] | suma=6326 | obciążenia=[169, 60, 0] | status=FEASIBLE | T=6.0319s
N= 40 | C=[170, 170, 170] | Wybrane=[7, 1, 0] |

## 3. Optymalizacja Tabu Search

Ruchy będą zdefiniowane w następujący sposób:
- wsadzenie przedmiotu z poza plecaka do dowolnego z nich,
- wyciągnięcie (usunięcie) przedmiotu z dowolnego z plecaków na zewnątrz

Pozwoli to tabu searchowi próbować dołożyć przedmioty lub je usunąć


In [4]:
basic_tabu_results = run_section_30_compare(section30_instances, section30_mz_results)

=== Porównanie MiniZinc i Tabu Search (ruchy add/remove, te same instancje) ===

N = 10
  MiniZinc | V=3805 | T=0.2899s | status=OPTIMAL
    Wybrane=[2, 4, 2] | Poza=2 | Wartości=[1230, 1345, 1230] | Obciążenia=[120, 167, 120]
    x = [3, 3, 1, 1, 2, 2, 0, 0, 2, 2]
    warunek_2+2+2+poza2 = true
  TS tabu=1 | V=3805 | T=0.0167s | status=HEURISTIC
    Wybrane=[2, 2, 4] | Poza=2 | Wartości=[1230, 1230, 1345] | Obciążenia=[120, 120, 167]
    x = [1, 3, 2, 1, 3, 2, 0, 0, 3, 3]
    warunek_2+2+2+poza2 = true
    Porównanie jakości: TS == MiniZinc
  TS tabu=2 | V=3805 | T=0.0008s | status=HEURISTIC
    Wybrane=[2, 2, 4] | Poza=2 | Wartości=[1230, 1230, 1345] | Obciążenia=[120, 120, 167]
    x = [1, 3, 2, 1, 3, 2, 0, 0, 3, 3]
    warunek_2+2+2+poza2 = true
    Porównanie jakości: TS == MiniZinc
  TS tabu=5 | V=3805 | T=0.0008s | status=HEURISTIC
    Wybrane=[2, 2, 4] | Poza=2 | Wartości=[1230, 1230, 1345] | Obciążenia=[120, 120, 167]
    x = [1, 3, 2, 1, 3, 2, 0, 0, 3, 3]
    warunek_2+2+2+po

Dict{Int64, Dict{Int64, SolverResult}} with 10 entries:
  50 => Dict(5=>SolverResult([1, 2, 1, 2, 3, 3, 0, 0, 3, 3  …  0, 0, 3, 0, 0, 0…
  15 => Dict(5=>SolverResult([2, 1, 1, 3, 2, 3, 0, 0, 1, 2, 1, 3, 3, 2, 3], 547…
  20 => Dict(5=>SolverResult([1, 3, 2, 2, 1, 3, 0, 0, 0, 2, 3, 1, 0, 2, 1, 0, 0…
  25 => Dict(5=>SolverResult([3, 1, 1, 2, 3, 2, 0, 0, 1, 3  …  2, 0, 3, 3, 0, 3…
  10 => Dict(5=>SolverResult([1, 3, 2, 1, 3, 2, 0, 0, 3, 3], 3805, 0.00081133, …
  35 => Dict(5=>SolverResult([3, 2, 1, 1, 3, 2, 0, 0, 1, 0  …  2, 1, 3, 2, 1, 0…
  45 => Dict(5=>SolverResult([1, 3, 1, 2, 3, 2, 0, 0, 3, 2  …  1, 0, 0, 3, 1, 1…
  55 => Dict(5=>SolverResult([2, 3, 3, 1, 1, 2, 0, 0, 0, 0  …  0, 0, 1, 0, 0, 0…
  30 => Dict(5=>SolverResult([3, 2, 3, 1, 2, 1, 0, 0, 3, 2  …  1, 3, 3, 2, 1, 2…
  40 => Dict(5=>SolverResult([3, 1, 2, 2, 1, 3, 0, 0, 2, 3  …  0, 0, 0, 0, 0, 0…

## Realizacja na 4.0
Dodajemy ruch polegający na przeniesieniu pomiędzy plecakami (`:transfer`). Uśredniamy 10 losowych instancji dla każdego z wymienionych rozmiarów.

In [5]:
exp40_results = run_section_40()

=== Wpływ ruchu transfer na Tabu Search (uśrednienie po 10 instancjach) ===
N=10 | tabu=1: takie samo (ΔV=0.0), wolniej (ΔT=+0.0003s) | tabu=2: takie samo (ΔV=0.0), wolniej (ΔT=+0.0192s) | tabu=5: takie samo (ΔV=0.0), szybciej (ΔT=-0.0001s)
N=15 | tabu=1: lepsze (ΔV=5.8), szybciej (ΔT=-0.0004s) | tabu=2: lepsze (ΔV=5.8), wolniej (ΔT=+0.0008s) | tabu=5: gorsze (ΔV=-1.9), wolniej (ΔT=+0.0001s)
N=20 | tabu=1: gorsze (ΔV=-2.4), wolniej (ΔT=+0.0002s) | tabu=2: gorsze (ΔV=-0.9), wolniej (ΔT=+0.0006s) | tabu=5: gorsze (ΔV=-4.8), szybciej (ΔT=-0.0001s)
N=25 | tabu=1: lepsze (ΔV=4.4), wolniej (ΔT=+0.0003s) | tabu=2: gorsze (ΔV=-1.1), wolniej (ΔT=+0.0005s) | tabu=5: gorsze (ΔV=-17.9), szybciej (ΔT=-0.0007s)
N=30 | tabu=1: gorsze (ΔV=-2.5), wolniej (ΔT=+0.0003s) | tabu=2: gorsze (ΔV=-7.5), wolniej (ΔT=+0.0001s) | tabu=5: gorsze (ΔV=-16.1), wolniej (ΔT=+0.0001s)
N=35 | tabu=1: lepsze (ΔV=4.1), szybciej (ΔT=-0.0003s) | tabu=2: lepsze (ΔV=4.1), wolniej (ΔT=+0.0001s) | tabu=5: gorsze (ΔV=-6.9), wolni

Dict{Int64, Dict{Int64, @NamedTuple{value::Float64, time::Float64}}} with 10 entries:
  50 => Dict(0=>(value = 9252.8, time = 2.03953), 5=>(value = 18745.6, time = 0…
  15 => Dict(0=>(value = 5870.9, time = 1.61183), 5=>(value = 5869.0, time = 0.…
  20 => Dict(0=>(value = 6622.5, time = 2.04032), 5=>(value = 7983.4, time = 0.…
  25 => Dict(0=>(value = 7167.4, time = 2.03321), 5=>(value = 9588.5, time = 0.…
  10 => Dict(0=>(value = 3804.3, time = 0.21252), 5=>(value = 3804.3, time = 0.…
  35 => Dict(0=>(value = 6562.0, time = 2.03535), 5=>(value = 13200.6, time = 0…
  45 => Dict(0=>(value = 8031.3, time = 2.03746), 5=>(value = 16307.4, time = 0…
  55 => Dict(0=>(value = 9927.5, time = 2.04281), 5=>(value = 20136.5, time = 0…
  30 => Dict(0=>(value = 5688.4, time = 2.03447), 5=>(value = 11003.4, time = 0…
  40 => Dict(0=>(value = 7617.0, time = 2.03418), 5=>(value = 15307.4, time = 0…

## Realizacja na 5.0
Algorytm Symulowanego Wyżarzania dla problemu plecakowego, bazujący na tym samym interfejsie ruchów co Tabu Search. Do implementacji włączamy trzy schematy chłodzenia.

In [6]:
run_section_50(exp40_results)


=== Symulowane wyżarzanie: porównanie z wcześniejszymi metodami ===
Dla każdego rozmiaru dobieram liczbę iteracji tak, aby czas SA był zbliżony do czasu TS z listą tabu długości 5.
N=10 | target TS(5)=0.0009s | dobrane iteracje SA=1000
N=15 | target TS(5)=0.0007s | dobrane iteracje SA=1000
N=20 | target TS(5)=0.0017s | dobrane iteracje SA=2000
N=25 | target TS(5)=0.0011s | dobrane iteracje SA=1000
N=30 | target TS(5)=0.0019s | dobrane iteracje SA=1000
N=35 | target TS(5)=0.0021s | dobrane iteracje SA=2000
N=40 | target TS(5)=0.0025s | dobrane iteracje SA=2000
N=45 | target TS(5)=0.0026s | dobrane iteracje SA=1000
N=50 | target TS(5)=0.0031s | dobrane iteracje SA=2000
N=55 | target TS(5)=0.0031s | dobrane iteracje SA=2000

| N | iteracje SA | MiniZinc | TS tabu=5 | SA linear | SA geometric | SA logarithmic |
|---|---|---|---|---|---|---|
| 10 | 1000 | V=3804.3, T=0.2125 | V=3804.3, T=0.0009 | V=3804.3, T=0.0006 | V=3804.3, T=0.0008 | V=3804.3, T=0.0007 |
| 15 | 1000 | V=5870.9, T=1.6118

## Wyniki:

Algorytm wyżarzania działa lepiej niz MiniZyc ale gorzej niż TabuSearch w porównywalnym czasie